In [1]:

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import os
import zipfile
import time
import copy
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

# Device configuration
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print('Device:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [2]:

input_folder = '/kaggle/input'
extract_folder = '/kaggle/working/Mango_Dataset'

os.makedirs(extract_folder, exist_ok=True)

# Find ZIP files
zip_files = []

for root, dirs, files in os.walk(input_folder):

    for file_name in files:

        if file_name.lower().endswith('.zip'):

            zip_files.append(
                os.path.join(root, file_name)
            )

# Extract ZIP files
for zip_path in zip_files:

    print('Extracting:', zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:

        zip_ref.extractall(extract_folder)

print('ZIP extraction completed!')


ZIP extraction completed!


In [3]:

expected_classes = [
    'Anthracnose',
    'Bacterial_Canker',
    'Cutting_Weevil',
    'Die_Back',
    'Gall_Midge',
    'Powdery_Mildew',
    'Sooty_Mould',
    'Healthy'
]

search_locations = [
    '/kaggle/input',
    '/kaggle/working/Mango_Dataset'
]

data_folder = None

image_extensions = (
    '.jpg', '.jpeg', '.png', '.bmp', '.webp'
)

# Find the folder containing all 8 classes
for search_folder in search_locations:

    for root, dirs, files in os.walk(search_folder):

        if set(expected_classes).issubset(set(dirs)):

            data_folder = root
            break

    if data_folder is not None:
        break

if data_folder is None:

    raise FileNotFoundError(
        'Could not find all 8 Mango class folders. '
        'Please check the Kaggle Input structure.'
    )

class_names = expected_classes

print('Dataset folder:', data_folder)
print('Classes:', class_names)
print('Number of classes:', len(class_names))

# Count images in each class
total_images = 0

for class_name in class_names:

    class_folder = os.path.join(
        data_folder, class_name
    )

    image_files = [
        file_name for file_name in os.listdir(class_folder)
        if file_name.lower().endswith(image_extensions)
    ]

    print(class_name, ':', len(image_files))

    total_images += len(image_files)

print('\nTotal images:', total_images)

assert len(class_names) == 8
assert total_images > 0


Dataset folder: /kaggle/input/datasets/srsanjana123/mango-leaf-disease-dataset/MLD24
Classes: ['Anthracnose', 'Bacterial_Canker', 'Cutting_Weevil', 'Die_Back', 'Gall_Midge', 'Powdery_Mildew', 'Sooty_Mould', 'Healthy']
Number of classes: 8
Anthracnose : 800
Bacterial_Canker : 800
Cutting_Weevil : 800
Die_Back : 800
Gall_Midge : 800
Powdery_Mildew : 800
Sooty_Mould : 800
Healthy : 800

Total images: 6400


In [4]:

tr_images, tr_targets = [], []
val_images, val_targets = [], []

np.random.seed(42)

for label, class_name in enumerate(class_names):

    class_folder = os.path.join(
        data_folder, class_name
    )

    image_files = sorted([
        os.path.join(class_folder, file_name)
        for file_name in os.listdir(class_folder)
        if file_name.lower().endswith(image_extensions)
    ])

    train_files, validation_files = train_test_split(
        image_files,
        test_size=0.20,
        random_state=42
    )

    tr_images.extend(train_files)
    tr_targets.extend([label] * len(train_files))

    val_images.extend(validation_files)
    val_targets.extend([label] * len(validation_files))

print('Training images:', len(tr_images))
print('Validation images:', len(val_images))


Training images: 5120
Validation images: 1280


In [5]:

image_size = 128

batch_size_no = 32
learning_rate = 1e-3
epoch_number = 10

weight_decay = 1e-4
dropout_rate = 0.3

scheduler_step_size = 5
scheduler_gamma = 0.5

number_of_classes = len(class_names)

print('Image size:', image_size)
print('Batch size:', batch_size_no)
print('Learning rate:', learning_rate)
print('Epochs:', epoch_number)
print('Weight decay:', weight_decay)
print('Dropout:', dropout_rate)
print('Scheduler step size:', scheduler_step_size)
print('Scheduler gamma:', scheduler_gamma)


Image size: 128
Batch size: 32
Learning rate: 0.001
Epochs: 10
Weight decay: 0.0001
Dropout: 0.3
Scheduler step size: 5
Scheduler gamma: 0.5


In [6]:

class MangoDataset(Dataset):

    def __init__(self, x, y):

        self.x = x
        self.y = y

    def __getitem__(self, ix):

        image_path = self.x[ix]

        with Image.open(image_path) as image:

            image = image.convert('RGB')

            image = image.resize(
                (image_size, image_size)
            )

            image = np.array(image)

        x = torch.tensor(image).float() / 255

        x = x.permute(2, 0, 1)

        y = torch.tensor(
            self.y[ix],
            dtype=torch.long
        )

        return x, y

    def __len__(self):

        return len(self.x)


In [7]:

def get_data():

    train = MangoDataset(
        tr_images,
        tr_targets
    )

    trn_dl = DataLoader(
        train,
        batch_size=batch_size_no,
        shuffle=True,
        num_workers=2,
        pin_memory=(device.type == 'cuda')
    )

    val = MangoDataset(
        val_images,
        val_targets
    )

    val_dl = DataLoader(
        val,
        batch_size=batch_size_no,
        shuffle=False,
        num_workers=2,
        pin_memory=(device.type == 'cuda')
    )

    return trn_dl, val_dl


trn_dl, val_dl = get_data()

print('Number of training batches:', len(trn_dl))
print('Number of validation batches:', len(val_dl))


Number of training batches: 160
Number of validation batches: 40


In [8]:

def get_model():

    model = nn.Sequential(

        # Feature Extractor / Convolution layers

        # 1st convolution
        nn.Conv2d(3, 40, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # 2nd convolution
        nn.Conv2d(40, 65, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # 3rd convolution
        nn.Conv2d(65, 70, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # Classifier
        nn.Flatten(),

        nn.Linear(70 * 16 * 16, 256),
        nn.ReLU(),

        # Dropout regularization
        nn.Dropout(dropout_rate),

        nn.Linear(256, number_of_classes)

    ).to(device)

    loss_fn = nn.CrossEntropyLoss()

    optimizer = Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    return model, loss_fn, optimizer


model, loss_fn, optimizer = get_model()

print(
    'Trainable parameters:',
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


Trainable parameters: 4655437


In [9]:

def train_one_epoch(data_loader, model, loss_fn, optimizer):

    model.train()

    epoch_losses = []

    for x, y in data_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        prediction = model(x)

        batch_loss = loss_fn(prediction, y)

        batch_loss.backward()

        optimizer.step()

        epoch_losses.append(batch_loss.item())

    return np.mean(epoch_losses)


In [10]:

@torch.no_grad()
def validation(data_loader, model, loss_fn):

    model.eval()

    val_losses = []

    correct_predictions = 0
    total_predictions = 0

    for x, y in data_loader:

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        prediction = model(x)

        batch_loss = loss_fn(prediction, y)

        val_losses.append(batch_loss.item())

        predicted_labels = prediction.argmax(dim=1)

        correct_predictions += (
            predicted_labels == y
        ).sum().item()

        total_predictions += y.size(0)

    val_loss = np.mean(val_losses)

    val_accuracy = (
        correct_predictions / total_predictions
    ) * 100

    return val_loss, val_accuracy


In [11]:

experiments = [

    {
        "name": "CNN 1 - Baseline",
        "batch_size": 32,
        "learning_rate": 0.001
    },

    {
        "name": "CNN 2 - Small Batch",
        "batch_size": 16,
        "learning_rate": 0.001
    },

    {
        "name": "CNN 3 - Large Batch",
        "batch_size": 64,
        "learning_rate": 0.001
    },

    {
        "name": "CNN 4 - Low Learning Rate",
        "batch_size": 32,
        "learning_rate": 0.0001
    },

    {
        "name": "CNN 5 - High Learning Rate",
        "batch_size": 32,
        "learning_rate": 0.01
    }

]

print('Total experiments:', len(experiments))

for exp in experiments:
    print(exp)


Total experiments: 5
{'name': 'CNN 1 - Baseline', 'batch_size': 32, 'learning_rate': 0.001}
{'name': 'CNN 2 - Small Batch', 'batch_size': 16, 'learning_rate': 0.001}
{'name': 'CNN 3 - Large Batch', 'batch_size': 64, 'learning_rate': 0.001}
{'name': 'CNN 4 - Low Learning Rate', 'batch_size': 32, 'learning_rate': 0.0001}
{'name': 'CNN 5 - High Learning Rate', 'batch_size': 32, 'learning_rate': 0.01}


In [12]:

all_results = []

for exp in experiments:

    print('\n' + '=' * 60)
    print('Starting:', exp["name"])
    print('=' * 60)

    # Set experiment parameters
    batch_size_no = exp["batch_size"]
    learning_rate = exp["learning_rate"]

    # Create DataLoaders
    trn_dl, val_dl = get_data()

    # Create a fresh model
    model, loss_fn, optimizer = get_model()

    # Learning rate scheduler
    scheduler = StepLR(
        optimizer,
        step_size=scheduler_step_size,
        gamma=scheduler_gamma
    )

    best_val_accuracy = 0.0
    best_epoch = 0

    final_train_loss = 0.0
    final_val_loss = 0.0

    start_time = time.time()

    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Training loop
    for epoch in range(epoch_number):

        train_loss = train_one_epoch(
            trn_dl,
            model,
            loss_fn,
            optimizer
        )

        val_loss, val_accuracy = validation(
            val_dl,
            model,
            loss_fn
        )

        # Save best validation accuracy
        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1

            best_model_weights = copy.deepcopy(
                model.state_dict()
            )

        final_train_loss = train_loss
        final_val_loss = val_loss

        print(
            f"Epoch [{epoch+1}/{epoch_number}] "
            f"Training Loss: {train_loss:.4f} | "
            f"Validation Loss: {val_loss:.4f} | "
            f"Validation Accuracy: {val_accuracy:.2f}%"
        )

        # Update learning rate
        scheduler.step()

    if device.type == 'cuda':
        torch.cuda.synchronize()

    training_time = time.time() - start_time

    # Save best model weights
    model.load_state_dict(best_model_weights)

    safe_name = (
        exp["name"]
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    torch.save(
        best_model_weights,
        f'/kaggle/working/{safe_name}_best.pth'
    )

    # Store experiment results
    all_results.append({

        "Experiment": exp["name"],

        "Batch Size": batch_size_no,

        "Initial Learning Rate": learning_rate,

        "Final Training Loss": final_train_loss,

        "Final Validation Loss": final_val_loss,

        "Best Validation Accuracy (%)": best_val_accuracy,

        "Best Epoch": best_epoch,

        "Training Time (seconds)": round(
            training_time, 2
        )

    })

    print('\nCompleted:', exp["name"])

    print(
        'Best Validation Accuracy:',
        round(best_val_accuracy, 2), '%'
    )

    print('Best Epoch:', best_epoch)

    print(
        'Training Time:',
        round(training_time, 2),
        'seconds'
    )

print('\nAll five experiments completed!')



Starting: CNN 1 - Baseline
Epoch [1/10] Training Loss: 1.4025 | Validation Loss: 0.9529 | Validation Accuracy: 63.59%
Epoch [2/10] Training Loss: 0.7618 | Validation Loss: 0.5282 | Validation Accuracy: 79.30%
Epoch [3/10] Training Loss: 0.4707 | Validation Loss: 0.4836 | Validation Accuracy: 82.42%
Epoch [4/10] Training Loss: 0.3292 | Validation Loss: 0.4192 | Validation Accuracy: 84.45%
Epoch [5/10] Training Loss: 0.2497 | Validation Loss: 0.2800 | Validation Accuracy: 90.31%
Epoch [6/10] Training Loss: 0.1575 | Validation Loss: 0.1760 | Validation Accuracy: 94.45%
Epoch [7/10] Training Loss: 0.1189 | Validation Loss: 0.1849 | Validation Accuracy: 93.75%
Epoch [8/10] Training Loss: 0.1003 | Validation Loss: 0.1297 | Validation Accuracy: 96.56%
Epoch [9/10] Training Loss: 0.0915 | Validation Loss: 0.1392 | Validation Accuracy: 96.09%
Epoch [10/10] Training Loss: 0.0713 | Validation Loss: 0.1254 | Validation Accuracy: 96.25%

Completed: CNN 1 - Baseline
Best Validation Accuracy: 96.56 